In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

# -------------------------------------------------------------
# 1. Load and Clean Dataset
# -------------------------------------------------------------
filepath = r'C:\Users\dell\OneDrive\Desktop\KMEC\PS Project\432_2024_5740_MOESM3_ESM.xlsx'

# Read sheet skipping non-data headers
raw_df = pd.read_excel(filepath, sheet_name='Results')
raw_df.columns = raw_df.iloc[0]
df = raw_df.iloc[1:].copy().reset_index(drop=True)

# Keep only rows with a valid target pair separated by '+'
df = df[df['Target(Gene Name)'].str.contains(r'\+', na=False)].copy()

# -------------------------------------------------------------
# 2. Extract & Normalize Target Pairs (Order-Invariant)
# -------------------------------------------------------------
def split_and_sort_targets(target_str):
    parts = [t.strip().upper() for t in str(target_str).split('+')]
    if len(parts) == 2:
        parts.sort()
        return parts[0], parts[1]
    return np.nan, np.nan

df[['Target_A', 'Target_B']] = df['Target(Gene Name)'].apply(
    lambda x: pd.Series(split_and_sort_targets(x))
)
df = df.dropna(subset=['Target_A', 'Target_B']).reset_index(drop=True)

# -------------------------------------------------------------
# 3. Label Creation
# -------------------------------------------------------------
# Approved = 1, otherwise (Phase 1-3, Discontinued, Blank, etc.) = 0
df['Target_Label'] = df['Drug Highest Phase'].apply(
    lambda x: 1 if pd.notna(x) and 'Approved' in str(x) else 0
)

print(f"Total valid target pairs: {len(df)}")
print(f"Target distribution:\n{df['Target_Label'].value_counts()}")

# -------------------------------------------------------------
# 4. Feature Encoding & XGBoost Model Pipeline
# -------------------------------------------------------------
X = df[['Target_A', 'Target_B']]
y = df['Target_Label']

# One-hot encode targets with handling for unseen pairs during inference
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), ['Target_A', 'Target_B'])
    ]
)

# Compute class imbalance weight
neg_count = (y == 0).sum()
pos_count = (y == 1).sum()
scale_pos_weight = neg_count / pos_count

clf = xgb.XGBClassifier(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    random_state=42
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', clf)
])

# -------------------------------------------------------------
# 5. Model Evaluation (Stratified K-Fold CV)
# -------------------------------------------------------------
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_roc_auc = cross_val_score(pipeline, X, y, cv=skf, scoring='roc_auc')
print(f"\n5-Fold Cross-Validation ROC-AUC: {cv_roc_auc.mean():.4f} (±{cv_roc_auc.std():.4f})")

# Fit on all available data for inference
pipeline.fit(X, y)

# -------------------------------------------------------------
# 6. Prediction Function for Any Target Pair
# -------------------------------------------------------------
def predict_pair(target_1: str, target_2: str):
    """
    Takes two targets, standardizes their order, and predicts 
    whether the combination is approved (1) or not (0).
    """
    pair = sorted([target_1.strip().upper(), target_2.strip().upper()])
    input_df = pd.DataFrame([{'Target_A': pair[0], 'Target_B': pair[1]}])
    
    pred = pipeline.predict(input_df)[0]
    prob = pipeline.predict_proba(input_df)[0][1]
    return {'Prediction': int(pred), 'Approval_Probability': round(float(prob), 4)}

# -------------------------------------------------------------
# 7. Take User Input
# -------------------------------------------------------------
if __name__ == "__main__":
    print("\n--- Enter Target Pair for Approval Prediction ---")
    user_target_1 = input("Enter Target 1 (e.g., BCMA): ")
    user_target_2 = input("Enter Target 2 (e.g., CD3): ")
    
    result = predict_pair(user_target_1, user_target_2)
    print(f"\nResult for {user_target_1.strip()} + {user_target_2.strip()}:")
    print(f"Output (Approved: 1, Not Approved: 0): {result['Prediction']}")
    print(f"Confidence / Probability: {result['Approval_Probability']}")

Total valid target pairs: 782
Target distribution:
Target_Label
0    770
1     12
Name: count, dtype: int64

5-Fold Cross-Validation ROC-AUC: 0.6776 (±0.1996)

--- Enter Target Pair for Approval Prediction ---

Result for BCMA + CD3:
Output (Approved: 1, Not Approved: 0): 1
Confidence / Probability: 0.8282
